In [31]:
from vllm import LLM, SamplingParams
from transformers import AutoProcessor

from tqdm import tqdm
import pandas as pd
import json

## Overview

In [2]:
df = pd.read_csv("ocr_extract.csv")
print(df.shape)
df.head()

(12167, 3)


,doc_id,พรรคการเมือง,คะแนน
0,constituency_10_1,NaN,NaN
1,constituency_10_10,NaN,NaN
2,constituency_10_10_page2,ประชาชน,"๔๑,๘๐๔ (สี่หมื่นหนึ่งพันแปดร้อยสี่)"
3,constituency_10_10_page2,เพื่อไทย,"๑๙,๐๔๗ (หนึ่งหมื่นเก้าพันสี่สิบเจ็ด)"
4,constituency_10_10_page2,โอกาสใหม่,"๙,๔๔๐ (เก้าพันสี่ร้อยสี่สิบ)"


In [65]:
prompt_template = '''## Task: Standardize Thai Election Data and Convert Thai Numerals to Arabic Integers.

## Input Data: You will receive a JSON object containing a doc_id, a พรรคการเมือง (Political Party), and คะแนน (Votes).

## Numeric Conversion
The คะแนน field contains Thai numerals (๑, ๒, ๓) or Thai word-numbers (เช่น สี่หมื่นหนึ่งพัน).
- Convert all Thai numerals and text into a standard Arabic Integer.
- Remove commas and whitespace.
- Example: "๔๑,๘๐๔ (สี่หมื่นหนึ่งพันแปดร้อยสี่)" -> 41804.

## Output Format: Return ONLY a JSON object with this structure:
{
"doc_id": "string",
"พรรคการเมือง": "string" or "null",
"คะแนน": integer
}'''

## Setup

In [4]:
# 1. Load model with multimodal support
model_id = "./typhoon2.5-qwen3-4b"
llm = LLM(
    model                  = model_id,
    gpu_memory_utilization = 0.8,
    max_model_len          = 8192,
    enforce_eager          = True,
)

INFO 03-22 05:17:16 [utils.py:238] non-default args: {'max_model_len': 8192, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'enforce_eager': True, 'model': './typhoon2.5-qwen3-4b'}
INFO 03-22 05:17:16 [model.py:531] Resolved architecture: Qwen3ForCausalLM
INFO 03-22 05:17:16 [model.py:1554] Using max model len 8192
INFO 03-22 05:17:17 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-22 05:17:17 [vllm.py:747] Asynchronous scheduling is enabled.
WARNING 03-22 05:17:17 [vllm.py:781] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 03-22 05:17:17 [vllm.py:792] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 03-22 05:17:19 [vllm.py:957] Cudagraph is disabled under eager mode
(EngineCore_DP0 pid=3202194) INFO 03-22 05:17:19 [core.py:101] Initializing a V1 LLM en

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore_DP0 pid=3202194) INFO 03-22 05:17:31 [default_loader.py:293] Loading weights took 5.95 seconds
(EngineCore_DP0 pid=3202194) INFO 03-22 05:17:31 [gpu_model_runner.py:4338] Model loading took 7.61 GiB memory and 7.939358 seconds
(EngineCore_DP0 pid=3202194) INFO 03-22 05:17:33 [gpu_worker.py:424] Available KV cache memory: 23.33 GiB
(EngineCore_DP0 pid=3202194) INFO 03-22 05:17:33 [kv_cache_utils.py:1314] GPU KV cache size: 169,856 tokens
(EngineCore_DP0 pid=3202194) INFO 03-22 05:17:33 [kv_cache_utils.py:1319] Maximum concurrency for 8,192 tokens per request: 20.73x
(EngineCore_DP0 pid=3202194) INFO 03-22 05:17:33 [core.py:282] init engine (profile, create kv cache, warmup model) took 1.71 seconds
(EngineCore_DP0 pid=3202194) WARNING 03-22 05:17:33 [vllm.py:781] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore_DP0 pid=3202194) WARNING 03-22 05:17:33 [vllm.py:792] Inductor compilation wa

In [66]:
processor = AutoProcessor.from_pretrained(model_id, use_fast=True)

filter_df = df[ ~df['คะแนน'].isnull() ]
conversations = []
for _, row in filter_df.iterrows():
    data = str({
        "doc_id": row['doc_id'],
        "พรรคการเมือง": row['พรรคการเมือง'],
        "คะแนน": row["คะแนน"]
    })

    messages = [
        {"role": "system", "content": prompt_template},
        {"role": "user", "content": data}
    ]
    prompt_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    conversations.append(prompt_text)

print(conversations[0])

<|im_start|>system
## Task: Standardize Thai Election Data and Convert Thai Numerals to Arabic Integers.

## Input Data: You will receive a JSON object containing a doc_id, a พรรคการเมือง (Political Party), and คะแนน (Votes).

## Numeric Conversion
The คะแนน field contains Thai numerals (๑, ๒, ๓) or Thai word-numbers (เช่น สี่หมื่นหนึ่งพัน).
- Convert all Thai numerals and text into a standard Arabic Integer.
- Remove commas and whitespace.
- Example: "๔๑,๘๐๔ (สี่หมื่นหนึ่งพันแปดร้อยสี่)" -> 41804.

## Output Format: Return ONLY a JSON object with this structure:
{
"doc_id": "string",
"พรรคการเมือง": "string" or "null",
"คะแนน": integer
}<|im_end|>
<|im_start|>user
{'doc_id': 'constituency_10_10_page2', 'พรรคการเมือง': 'ประชาชน', 'คะแนน': '๔๑,๘๐๔ (สี่หมื่นหนึ่งพันแปดร้อยสี่)'}<|im_end|>
<|im_start|>assistant



## Inference

In [67]:
# 3. Generate (batched version accepts list of conversations)
sampling_params = SamplingParams(
    temperature=0.0, top_p=1, top_k=-1,
    max_tokens=8192,
    skip_special_tokens=True,
)

outputs = llm.generate(conversations, sampling_params=sampling_params)

results = []
for o in outputs:
    generated_text = o.outputs[0].text
    results.append(generated_text)

print(results[0])

Rendering prompts:   0%|          | 0/9895 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/9895 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

{
"doc_id": "constituency_10_10_page2",
"พรรคการเมือง": "ประชาชน",
"คะแนน": 41804
}


In [69]:
def process_results_to_df(results_list):
    data = [json.loads(item) for item in results_list]
    df = pd.DataFrame(data)
    
    return df

# Execute
df_final = process_results_to_df(results)
df_final.head()

,doc_id,พรรคการเมือง,คะแนน
0,constituency_10_10_page2,ประชาชน,41804
1,constituency_10_10_page2,เพื่อไทย,19047
2,constituency_10_10_page2,โอกาสใหม่,9440
3,constituency_10_10_page2,ภูมิใจไทย,7925
4,constituency_10_10_page2,ประชาธิปัตย์,6372


In [76]:
original = df.loc[ ~df.index.isin(filter_df.index)]
submit_df = pd.concat([original, df_final], ignore_index=True).reset_index(drop=True)
submit_df['document'] = submit_df['doc_id'].apply(lambda x: x.split("_page")[0])
submit_df = submit_df[ ~(submit_df['พรรคการเมือง'] == 'null') ].reset_index(drop=True)

submit_dict = {}
for doc in tqdm(submit_df['document'].unique()):
    temp_df = submit_df.loc[ submit_df['document'] == doc ]
    temp_df = temp_df.drop(columns='พรรคการเมือง').dropna()
    submit_dict[doc] = temp_df['คะแนน'].tolist()

100%|██████████| 300/300 [00:00<00:00, 806.92it/s]


## Submission

In [77]:
submission = pd.read_csv("final_data/sample_submission.csv")
submission['doc'] = submission['id'].apply(lambda x: "_".join(x.split("_")[:-1]))


# 1. Prepare a list to collect results
all_votes = []

for doc in tqdm(submission['doc'].unique()):
    # Get the indices of the rows belonging to this doc
    doc_indices = submission[submission['doc'] == doc].index
    num_rows = len(doc_indices)
    
    answer = submit_dict.get(doc, []) # Use .get to avoid KeyErrors
    
    # Handle length logic
    if len(answer) > num_rows:
        final_values = answer[:num_rows]
    elif len(answer) < num_rows:
        final_values = answer + [0] * (num_rows - len(answer))
    else:
        final_values = answer
        
    # Create a Series with the correct original indices
    doc_votes = pd.Series(final_values, index=doc_indices)
    all_votes.append(doc_votes)

# 2. Combine all parts and assign back to the main dataframe once
submission['votes'] = pd.concat(all_votes)
submission.head()

100%|██████████| 300/300 [00:00<00:00, 1374.62it/s]


,id,party_name (????????????????????????),votes,doc
0,constituency_10_1_1,ประชาธิปัตย์,34167,constituency_10_1
1,constituency_10_1_2,ภูมิใจไทย,14813,constituency_10_1
2,constituency_10_1_3,เศรษฐกิจ,14368,constituency_10_1
3,constituency_10_1_4,กล้าธรรม,6030,constituency_10_1
4,constituency_10_1_5,พลวัต,2075,constituency_10_1


In [78]:
submission.iloc[:, [0, 2]].to_csv("typhoon_no-null.csv", index=False, encoding='utf-8-sig')